<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》</a> 一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


- 如需安装这个 bonus Notebook 的额外依赖，请取消下面单元格的注释并运行：

In [ ]:
# pip install -r requirements-extra.txt

  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   -------- ------------------------------- 2.4/10.8 MB 15.0 MB/s eta 0:00:01
   ------------------------- -------------- 6.8/10.8 MB 19.1 MB/s eta 0:00:01
   ------------------------------------- -- 10.2/10.8 MB 17.7 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 16.8 MB/s  0:00:00
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 663.6/663.6 kB 13.0 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------  3.9/4.0 MB 19.5 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 18.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ?


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 比较多种字节对编码（BPE）实现

<br>
&nbsp;

## 1. 使用 `tiktoken` 中的 BPE

In [2]:
from importlib.metadata import version

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.13.0


In [3]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. Is this-- a test?"

In [4]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [5]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


In [6]:
print(tik_tokenizer.n_vocab)

50257


<br>
&nbsp;

## 2. 使用 GPT-2 采用的原始 BPE 实现

In [ ]:
from bpe_openai_gpt2 import get_encoder, download_vocab# 导入 bpe_openai_gpt2 库，用于处理 byte pair encoding

In [8]:
download_vocab()

Fetching encoder.json: 1.04Mit [00:00, 3.69Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 2.53Mit/s]                                                       


In [9]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [10]:
integers = orig_tokenizer.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [11]:
strings = orig_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


<br>
&nbsp;

## 3. 通过 Hugging Face transformers 使用 BPE

In [12]:
import transformers

transformers.__version__

/Users/sebastian/Developer/LLMs-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'4.49.0'

In [13]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [14]:
hf_tokenizer(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [15]:
from transformers import GPT2TokenizerFast

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [16]:
hf_tokenizer_fast(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

<br>
&nbsp;

## 4. 使用我自己从零实现的 BPE tokenizer

In [17]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # 加载 Notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # 创建一个模块，用于存放导入的函数和类
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # 遍历 Notebook 单元格，只执行函数或类定义
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # 检查是否为函数或类定义
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [18]:
imported_module = import_from_notebook()
BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [19]:
integers = tokenizer_gpt2.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


<br>
&nbsp;

## 5. 快速性能基准测试

In [20]:
with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

&nbsp;
### 5.1 原始 OpenAI GPT-2 tokenizer

In [21]:
%timeit orig_tokenizer.encode(raw_text)

3.84 ms ± 9.83 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.2 Tiktoken OpenAI GPT-2 tokenizer

In [22]:
%timeit tik_tokenizer.encode(raw_text)

901 μs ± 6.27 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


&nbsp;
### 5.3 Hugging Face OpenAI GPT-2 tokenizer

In [23]:
%timeit hf_tokenizer(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


11 ms ± 94.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [24]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

10.8 ms ± 180 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


3.66 ms ± 3.67 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [26]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

3.77 ms ± 49.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.4 我自己实现的 GPT-2 tokenizer（用于教学目的）

In [27]:
%timeit tokenizer_gpt2.encode(raw_text)

9.37 ms ± 50.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
